# Data Cleaning — accidentologie0.csv

In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv("../data/accidentologie0.csv", sep=";")
print(f"Original shape: {df.shape}")
df.head(3)

Original shape: (33551, 30)


,Date,IdUsager,PV,Arrondissement,Mode,Catégorie,Gravité,Age,Genre,Milieu,...,Nom arrondissement,Arrondissement.1,arrondgeo,Adresse 1,Adresse 2,Coordonnées.1,Nom arrondissement.1,Coordonnées.2,Nom arrondissement.2,Arrondissement.2
0,2024-10-16,9630806,7435,75120,2 Roues Motorisées,Conducteur,Blessé léger,28.0,Masculin,En-Agg,...,NaN,75120,"{""coordinates"": [[[[2.415286456, 48.855177828]...",AVENUE DU DOCTEUR GLEY,RUE DES FRERES FLAVIEN,"48.876801, 2.411995",Paris 20e Arrondissement,"48.876801, 2.411995",Paris 20e Arrondissement,75120.0
1,2024-10-19,9647261,7523,75111,2 Roues Motorisées,Conducteur,Blessé léger,24.0,Masculin,En-Agg,...,NaN,75111,"{""coordinates"": [[[[2.369123881, 48.853166231]...",Rue Mercoeur,Rue Léon Frot,"48.857625, 2.384619",Paris 11e Arrondissement,"48.857625, 2.384619",Paris 11e Arrondissement,75111.0
2,2024-10-26,9654580,7725,75110,EDP-m,Conducteur,Blessé léger,20.0,Masculin,En-Agg,...,NaN,75110,"{""coordinates"": [[[[2.363872117, 48.867516287]...",Rue de la Fidélité,BOULEVARD DE MAGENTA,"48.874804, 2.357233",Paris 10e Arrondissement,"48.874804, 2.357233",Paris 10e Arrondissement,75110.0


## 1. Coalesce duplicate columns

Fill nulls in primary columns from their `.1` duplicates before dropping.

In [4]:
df["Coordonnées"] = df["Coordonnées"].fillna(df["Coordonnées.1"])
df["Nom arrondissement"] = df["Nom arrondissement"].fillna(df["Nom arrondissement.1"])

print(f"Coordonnées nulls after fill: {df['Coordonnées'].isnull().sum()}")
print(f"Nom arrondissement nulls after fill: {df['Nom arrondissement'].isnull().sum()}")

Coordonnées nulls after fill: 0
Nom arrondissement nulls after fill: 0


## 2. Drop redundant columns

In [5]:
cols_to_drop = [
    "Arrondissement.1",
    "Arrondissement.2",
    "Coordonnées.1",
    "Coordonnées.2",
    "Nom arrondissement.1",
    "Nom arrondissement.2",
    "Adresse 2",
    "Blessés Légers",
    "Blessés hospitalisés",
    "Tué",
]
df.drop(columns=cols_to_drop, inplace=True)
print(f"Shape after drop: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Shape after drop: (33551, 20)
Remaining columns: ['Date', 'IdUsager', 'PV', 'Arrondissement', 'Mode', 'Catégorie', 'Gravité', 'Age', 'Genre', 'Milieu', 'Adresse', 'Longitude', 'Latitude', 'Id accident', "Tranche d'age", 'Résumé', 'Coordonnées', 'Nom arrondissement', 'arrondgeo', 'Adresse 1']


## 3. Fix data types & formatting

In [6]:
# Longitude / Latitude: comma decimal -> dot decimal
df["Longitude"] = pd.to_numeric(df["Longitude"].astype(str).str.replace(",", "."), errors="coerce")
df["Latitude"] = pd.to_numeric(df["Latitude"].astype(str).str.replace(",", "."), errors="coerce")

# Date -> datetime
df["Date"] = pd.to_datetime(df["Date"])

# Age -> nullable int
df["Age"] = df["Age"].astype("Int64")

# Fix inconsistent spacing in Tranche d'age
df["Tranche d'age"] = df["Tranche d'age"].str.replace(r"(\d+)ans", r"\1 ans", regex=True)

print("dtypes:")
print(df.dtypes)

dtypes:
Date                  datetime64[us]
IdUsager                       int64
PV                             int64
Arrondissement                 int64
Mode                             str
Catégorie                        str
Gravité                          str
Age                            Int64
Genre                            str
Milieu                           str
Adresse                          str
Longitude                    float64
Latitude                     float64
Id accident                    int64
Tranche d'age                    str
Résumé                           str
Coordonnées                      str
Nom arrondissement               str
arrondgeo                        str
Adresse 1                        str
dtype: object


## 4. Extract features from Résumé

The `Résumé` column contains structured French text. We parse it into 8 new columns.

In [7]:
resume = df["Résumé"].fillna("")

# --- Type d'accident ---
df["type_accident"] = resume.str.extract(
    r"(Accident (?:Léger|Grave) non mortel|Accident Mortel)", expand=False
)

# --- Type d'intersection ---
df["type_intersection"] = resume.str.extract(
    r",\s*(En X|En T|En Y|Place|En giratoire|Hors intersection)", expand=False
)

# --- Éclairage ---
def parse_eclairage(text):
    if "Nuit" in text:
        if "non allumé" in text or "non allume" in text:
            return "Nuit non éclairée"
        return "Nuit éclairée"
    if "Crépuscule" in text or "crepuscule" in text or "aube" in text:
        return "Crépuscule"
    if "Plein jour" in text or "plein jour" in text:
        return "Plein jour"
    return None

df["eclairage"] = resume.apply(parse_eclairage)

# --- Météo ---
df["meteo"] = resume.str.extract(
    r"météo\s+(Normale|Pluie légère|Pluie forte|Temps couvert|Neige|Brouillard|Autre)",
    expand=False
)

# --- État de surface ---
df["etat_surface"] = resume.str.extract(
    r"surface chaussée\s*:\s*(Normale|Mouillée|Verglacée|Enneigée|Corps gras|Inondée|Autre)",
    expand=False
)

# --- VMA (speed limit) ---
df["vma"] = resume.str.extract(r"VMA à\s*(\d+)", expand=False).astype("Int64")

# --- Filter VMA to realistic Paris values (drop parsing artifacts 1-6, 35-45, 60, 80, 300) ---
valid_vma = {10, 15, 20, 25, 30, 50, 70, 90, 110}
invalid_vma = df["vma"].notna() & (~df["vma"].isin(valid_vma))
n_invalid = invalid_vma.sum()
df.loc[invalid_vma, "vma"] = pd.NA
print(f"VMA: set {n_invalid} invalid values to NaN (kept {df['vma'].notna().sum()} valid)")

# --- Type de route ---
def parse_type_route(text):
    m = re.search(r"circulant sur\s+(.+?)\s+\(VMA", text)
    if m:
        raw = m.group(1).strip()
        if "Voie Communale" in raw or "VC" in raw:
            return "Voie Communale"
        if "Autoroute" in raw or "A " in raw:
            return "Autoroute"
        if "Route Départementale" in raw or "RD" in raw:
            return "Route Départementale"
        if "Route de métropole" in raw or "RM" in raw:
            return "Route Métropolitaine"
        if "stationnement" in raw.lower():
            return "Stationnement"
        if "Hors réseau" in raw:
            return "Hors réseau public"
        return "Autre"
    return None

df["type_route"] = resume.apply(parse_type_route)

# --- Partie adverse (who was hit) ---
def parse_partie_adverse(text):
    m = re.search(r"heurte\s+\d+\s+(.+)", text)
    if m:
        raw = m.group(1).strip()
        if "Piéton" in raw or "Pieton" in raw or "piéton" in raw or "pieton" in raw:
            return "Piéton"
        if "Bicyclette" in raw or "bicyclette" in raw:
            return "Bicyclette"
        if "Moto" in raw or "moto" in raw:
            return "Moto"
        if "Scooter" in raw or "scooter" in raw:
            return "Scooter"
        if "Vélo" in raw or "velo" in raw or "Velo" in raw:
            return "Vélo"
        if "VT" in raw or "tourisme" in raw:
            return "VT"
        if "VU" in raw:
            return "VU"
        if "3 RM" in raw or "3rm" in raw:
            return "3RM"
        return "Autre"
    return None

df["partie_adverse"] = resume.apply(parse_partie_adverse)

# Coverage report
new_cols = ["type_intersection", "eclairage", "meteo",
            "etat_surface", "vma", "type_route", "partie_adverse"]
coverage = pd.DataFrame({
    "parsed": [df[c].notna().sum() for c in new_cols],
    "null": [df[c].isnull().sum() for c in new_cols],
}, index=new_cols)
coverage["pct"] = (coverage["parsed"] / len(df) * 100).round(1)
coverage

VMA: set 54 invalid values to NaN (kept 29497 valid)


,parsed,null,pct
type_intersection,28811,4740,85.9
eclairage,29588,3963,88.2
meteo,29425,4126,87.7
etat_surface,29541,4010,88.0
vma,29497,4054,87.9
type_route,29553,3998,88.1
partie_adverse,10263,23288,30.6


In [8]:
df.drop(columns=["Résumé", "type_accident"], inplace=True)
print(f"Shape after dropping Résumé & type_accident: {df.shape}")

Shape after dropping Résumé & type_accident: (33551, 26)


## 5. Rename columns for consistency

In [9]:
df.rename(columns={
    "Nom arrondissement": "nom_arrondissement",
    "IdUsager": "id_usager",
    "Id accident": "id_accident",
    "Tranche d'age": "tranche_age",
    "Arrondissement": "arrondissement",
    "Adresse 1": "adresse_secondaire",
}, inplace=True)

df.columns.tolist()

['Date',
 'id_usager',
 'PV',
 'arrondissement',
 'Mode',
 'Catégorie',
 'Gravité',
 'Age',
 'Genre',
 'Milieu',
 'Adresse',
 'Longitude',
 'Latitude',
 'id_accident',
 'tranche_age',
 'Coordonnées',
 'nom_arrondissement',
 'arrondgeo',
 'adresse_secondaire',
 'type_intersection',
 'eclairage',
 'meteo',
 'etat_surface',
 'vma',
 'type_route',
 'partie_adverse']

## 6. Final validation

In [10]:
print(f"Final shape: {df.shape}")
df.head(5)

Final shape: (33551, 26)


,Date,id_usager,PV,arrondissement,Mode,Catégorie,Gravité,Age,Genre,Milieu,...,nom_arrondissement,arrondgeo,adresse_secondaire,type_intersection,eclairage,meteo,etat_surface,vma,type_route,partie_adverse
0,2024-10-16,9630806,7435,75120,2 Roues Motorisées,Conducteur,Blessé léger,28,Masculin,En-Agg,...,Paris 20e Arrondissement,"{""coordinates"": [[[[2.415286456, 48.855177828]...",AVENUE DU DOCTEUR GLEY,En X,Plein jour,Normale,Normale,30,Voie Communale,Moto
1,2024-10-19,9647261,7523,75111,2 Roues Motorisées,Conducteur,Blessé léger,24,Masculin,En-Agg,...,Paris 11e Arrondissement,"{""coordinates"": [[[[2.369123881, 48.853166231]...",Rue Mercoeur,En X,Nuit éclairée,Normale,Normale,30,Voie Communale,NaN
2,2024-10-26,9654580,7725,75110,EDP-m,Conducteur,Blessé léger,20,Masculin,En-Agg,...,Paris 10e Arrondissement,"{""coordinates"": [[[[2.363872117, 48.867516287]...",Rue de la Fidélité,En X,Plein jour,Normale,Normale,30,Voie Communale,Autre
3,2024-11-26,9680888,8373,75119,Piéton,Piéton,Blessé léger,26,Masculin,En-Agg,...,Paris 19e Arrondissement,"{""coordinates"": [[[[2.410820319, 48.878436176]...",RUE DU MAROC,En T,Plein jour,Normale,Normale,30,Voie Communale,Autre
4,2024-11-29,9682542,8467,75114,Vélo,Conducteur,Blessé léger,21,Masculin,En-Agg,...,Paris 14e Arrondissement,"{""coordinates"": [[[[2.336656992, 48.839671108]...",BOULEVARD JOURDAN,Hors intersection,Plein jour,Normale,Normale,50,Voie Communale,NaN


In [11]:
df.dtypes.to_frame("dtype")

,dtype
Date,datetime64[us]
id_usager,int64
PV,int64
arrondissement,int64
Mode,str
Catégorie,str
Gravité,str
Age,Int64
Genre,str
Milieu,str


In [12]:
nulls = df.isnull().sum().to_frame("null_count")
nulls["pct"] = (nulls["null_count"] / len(df) * 100).round(1)
nulls.sort_values("null_count", ascending=False)

,null_count,pct
adresse_secondaire,23317,69.5
partie_adverse,23288,69.4
type_intersection,4740,14.1
meteo,4126,12.3
vma,4054,12.1
etat_surface,4010,12.0
type_route,3998,11.9
eclairage,3963,11.8
Adresse,3,0.0
Date,0,0.0


In [13]:
for col in ["type_intersection", "eclairage", "meteo",
            "etat_surface", "type_route", "partie_adverse"]:
    print(f"{col}: {df[col].dropna().unique()}")

type_intersection: <StringArray>
['En X', 'En T', 'Hors intersection', 'En Y', 'Place', 'En giratoire']
Length: 6, dtype: str
eclairage: <StringArray>
['Plein jour', 'Nuit éclairée', 'Crépuscule', 'Nuit non éclairée']
Length: 4, dtype: str
meteo: <StringArray>
[      'Normale',  'Pluie légère', 'Temps couvert',   'Pluie forte',
         'Autre',         'Neige',    'Brouillard']
Length: 7, dtype: str
etat_surface: <StringArray>
[   'Normale',   'Mouillée', 'Corps gras',      'Autre',   'Enneigée',
  'Verglacée',    'Inondée']
Length: 7, dtype: str
type_route: <StringArray>
[      'Voie Communale',            'Autoroute', 'Route Métropolitaine',
        'Stationnement',                'Autre',   'Hors réseau public',
 'Route Départementale']
Length: 7, dtype: str
partie_adverse: <StringArray>
['Moto', 'Autre', '3RM', 'VT', 'Vélo', 'VU', 'Piéton', 'Scooter',
 'Bicyclette']
Length: 9, dtype: str


## 7. Save cleaned dataset

In [14]:
df.to_csv("../data/accidentologie0_clean.csv", index=False)
print(f"Saved {df.shape[0]} rows x {df.shape[1]} columns to data/accidentologie0_clean.csv")

Saved 33551 rows x 26 columns to data/accidentologie0_clean.csv
